In [1]:
import tensorflow as tf
from pathlib import Path

In [7]:
tf.random.set_seed(123)

project_dir = Path.cwd().parent
img_dir = project_dir / "img"

img_height = 180
img_width = 180
batch_size = 32
seed = 123

train_dataset = tf.keras.utils.image_dataset_from_directory(
    img_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    img_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_dataset.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

""" 
I created another notebook to give EfficientNetB0 architecture a try but there seems 
to be an issue with my versions of tf and keras as I tried many different tweaks and 
kept getting a shape mismatch error, so the MobileNetV2 architecture below seems 
like the best fit for my small dataset if I want to use transfer learning
"""
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    tf.keras.layers.RandomFlip('horizontal'),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15
)

val_loss, val_acc = model.evaluate(validation_dataset, verbose=2)

model.save('../models/mobilenetv2_model.keras')

Found 197 files belonging to 3 classes.
Using 158 files for training.
Found 197 files belonging to 3 classes.
Using 39 files for validation.


/var/folders/04/s_rk8gfn6vq1608hxwwtl_040000gn/T/ipykernel_2509/2883608252.py:42: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


Epoch 1/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 7s 721ms/step - accuracy: 0.3924 - loss: 2.0754 - val_accuracy: 0.4103 - val_loss: 1.5478
Epoch 2/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 143ms/step - accuracy: 0.4304 - loss: 1.5911 - val_accuracy: 0.7436 - val_loss: 0.7036
Epoch 3/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.6076 - loss: 1.3225 - val_accuracy: 0.4872 - val_loss: 2.0778
Epoch 4/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 130ms/step - accuracy: 0.6519 - loss: 1.2213 - val_accuracy: 0.6410 - val_loss: 1.0525
Epoch 5/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 111ms/step - accuracy: 0.6266 - loss: 1.1443 - val_accuracy: 0.7692 - val_loss: 0.6152
Epoch 6/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step - accuracy: 0.6772 - loss: 1.0167 - val_accuracy: 0.7436 - val_loss: 0.8642
Epoch 7/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 107ms/step - accuracy: 0.6582 - loss: 1.0761 - val_accuracy: 0.7436 - val_loss: 0.9067
Epoch 8/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.7278 - loss: 0.9087 - val_accuracy: 0.7179 - val_loss: